# 04b – Objects in practice

Three ideas from OOP that are worth remembering:

1. **A class keeps data and functions together.** One object knows everything about itself.
2. **The same method on different things.** If every shape has `area()`, one loop handles them all.
3. **Your objects can behave like built-in types.** `print(F)` is readable, `F1 + F2` works.

And one rule on top: **if a function is enough, write a function.**

## 0. You have been using objects all along

Before running the cell: what will it print?

In [ ]:
numbers = [1, 2]
numbers.append(3)          # the dot = a method of the object 'numbers'
print(numbers)
print(type(5), type("hello"), type(numbers))

**Guess first:** what do these lines print?

In [ ]:
print(2 + 3)
print("2" + "3")
print(len("hello"), len([1, 2, 3]))

The same `+` does different things. That is because `+` is just a method,
and every class writes its own version of it:

In [ ]:
print((2).__add__(3))
print("2".__add__("3"))

## 1. Forces that add up like vectors

A force has two components. First attempt:

In [ ]:
class Force:
    def __init__(self, x, y):
        self.x = x
        self.y = y


F = Force(3, 0)
print(F)

That is not very readable. Let us teach `Force` three *magic methods*:
`__repr__` (how to print it), `__add__` (what `+` does) and `__abs__` (what `abs()` does).

In [ ]:
import math


class Force:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Force({self.x}, {self.y})"

    def __add__(self, other):
        return Force(self.x + other.x, self.y + other.y)

    def __abs__(self):
        return math.hypot(self.x, self.y)


F = Force(3, 0) + Force(0, 4)
print(F)
print(abs(F))

Because `+` works, even the built-in `sum()` works – the resultant of many forces:

In [ ]:
forces = [Force(3, 0), Force(0, 4), Force(-1, -1)]
R = sum(forces, Force(0, 0))
print(R, abs(R))

**Your turn:** make `F * 2` double the force (method `__mul__`).

In [ ]:
class ScalableForce(Force):      # inherits everything from Force
    def __mul__(self, k):
        ...                           # TODO: return a new Force


print(ScalableForce(3, 4) * 2)      # should print Force(6, 8)

## 2. Python uses inheritance too

**Guess first:** what is `True + True`?

In [ ]:
print(True + True)
print(isinstance(True, int))
print(bool.__bases__)       # the parent class of bool

`bool` inherits from `int`: `True` behaves like `1`, `False` like `0`.
That is why `sum([True, False, True])` counts the `True` values.

## 3. Mystery: why does Bob suddenly have Matlab?

In [ ]:
class Student:
    courses = ['OOP', 'Mathematics']    # default courses for every student

    def __init__(self, name):
        self.name = name


alice = Student('Alice')
bob = Student('Bob')
alice.courses.append('Matlab')
print(bob.courses)

**Discuss:** what happened? A hint:

In [ ]:
print(alice.courses is bob.courses)     # is it the very same list?

`courses` belongs to the **class**, so all students share one list.
Fix: create the list in `__init__`, so every **instance** gets its own.

In [ ]:
class Student:
    def __init__(self, name):
        self.name = name
        self.courses = ['OOP', 'Mathematics']   # a new list for each student


alice = Student('Alice')
bob = Student('Bob')
alice.courses.append('Matlab')
print(bob.courses)

## 4. Is a tube a circle?

A circle and its second moment of area $I = \pi r^4 / 4$:

In [ ]:
class Circle:
    def __init__(self, r):
        self.r = r

    def __repr__(self):
        return f"Circle({self.r})"

    def area(self):
        return math.pi * self.r**2

    def I(self):                      # second moment of area
        return math.pi * self.r**4 / 4

A tube is round too, so let it inherit from `Circle` and fix only the area:

In [ ]:
class Tube(Circle):                   # "a tube is a circle"?
    def __init__(self, R, r):
        super().__init__(R)
        self.r_in = r

    def area(self):
        return math.pi * (self.r**2 - self.r_in**2)


t = Tube(10, 8)
print(t.area())
print(t.I())

**Check by hand:** for a tube $I = \pi (R^4 - r^4) / 4$. What should `t.I()` be?

No error, but a wrong number: the tube silently inherited `I()` of a **full** circle.
A tube *is not* a circle. A tube *has* two circles – outer and inner:

In [ ]:
class Tube:                           # no parent – a tube HAS two circles
    def __init__(self, R, r):
        self.outer = Circle(R)
        self.inner = Circle(r)

    def __repr__(self):
        return f"Tube({self.outer.r}, {self.inner.r})"

    def area(self):
        return self.outer.area() - self.inner.area()

    def I(self):
        return self.outer.I() - self.inner.I()


t = Tube(10, 8)
print(t.area())
print(t.I())

**Rule:** inherit when "A *is* B" holds completely. When "A *has* B", put B inside A.
If a child would inherit something that is not true for it, do not inherit.

## 5. Same method, different things – no common parent needed

In [ ]:
class Rectangle:
    def __init__(self, b, h):
        self.b = b
        self.h = h

    def __repr__(self):
        return f"Rectangle({self.b}, {self.h})"

    def area(self):
        return self.b * self.h

    def I(self):
        return self.b * self.h**3 / 12

`Circle`, `Tube` and `Rectangle` have no common parent. But all of them have `area()` and `I()`,
so one loop works for all of them. Python calls this *duck typing*:
"if it quacks like a duck, it is a duck".

$I / A$ says how stiff in bending a section is per kilogram of material. **Guess first:** which one wins?

In [ ]:
sections = [Circle(10), Tube(10, 8), Rectangle(10, 40), Rectangle(40, 10)]

for s in sections:
    print(f"{s!r:18} I/A = {s.I() / s.area():6.1f}")

The same plank, only rotated: 16× difference. That is why joists stand on edge.

**Your turn:** a thin tube `Tube(10, 9)` and a full circle with **the same area** (same mass).
Which one is stiffer and how many times?

In [ ]:
thin = Tube(10, 9)
full = ...      # TODO: a Circle with the same area as 'thin'

print(thin.area(), full.area())
print(thin.I() / full.I())

## 6. Let AI write it, you check it

Ask an AI assistant:

> Write a Python class `IBeam(h, b, tw, tf)` with methods `area()` and `I()`
> (second moment of area about the strong axis), in the same style as the `Circle` class above.

Paste its answer into the next cell.

In [ ]:
# paste the AI answer here

**Check 1 – a limit case.** If the web is as wide as the flanges (`tw = b`),
the I-beam becomes a full rectangle. No table needed:

In [ ]:
print(IBeam(200, 100, 100, 8.5).I(), Rectangle(100, 200).I())

**Check 2 – the steel table.** IPE 200: h = 200, b = 100, t<sub>w</sub> = 5.6, t<sub>f</sub> = 8.5 mm
→ A = 28.5 cm², I<sub>y</sub> = 1943 cm⁴.

In [ ]:
beam = IBeam(200, 100, 5.6, 8.5)       # mm
print(f"A = {beam.area() / 100:.1f} cm²   (table: 28.5)")
print(f"I = {beam.I() / 1e4:.0f} cm⁴    (table: 1943)")

**Discuss:** about 5 % off. Is the AI wrong?

Not necessarily: a real IPE has rounded corners between the web and the flanges (r = 12 mm),
our model ignores them. Deciding whether 5 % matters is an engineering decision, not a programming one.

## Take-away

- **Code is cheap, the model is not.** AI writes a class in five seconds. Whether a tube is a circle, and what the
  model may ignore, is your decision.
- **An interface is a contract.** "Every section has `area()` and `I()`." You can check a contract even on code you
  did not write: limit cases, hand calculations, tables.
- **Read more than you write.** Most code you will meet will be someone else's or AI's.

Even Linus Torvalds, a famous critic of C++ and its object models, uses objects in the Linux kernel
(`struct file_operations` – every driver fills in its own `read`, `write`, …).
He does not criticise objects, he criticises badly chosen models.

> **AI writes the code. The engineer designs the model and checks it.**